In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import re
from collections import Counter
from sklearn.model_selection import train_test_split
import time

In [ ]:
# 1. Load & Preprocess
df = pd.read_csv('all_songs_data_50k.csv')
song_df = pd.read_csv('songs.csv')

In [17]:
# --- Konfigurasi Model & Pelatihan ---
CONFIG = {
    # Dimensi Embedding untuk Teks
    "text_embedding_dim": 150, # dinaikkan
    "hidden_dim": 300, # dinaikkan
    "n_layers": 2,
    "dropout": 0.5, # disesuaikan
    # Dimensi Output Embedding (HARUS SAMA untuk kedua menara)
    "output_embedding_dim": 64,
    "batch_size": 128,
    "learning_rate": 0.001,
    "epochs": 25, # dinaikkan untuk konvergensi yang lebih baik
    "max_len": 50,
    "margin": 0.2,
    "min_song_freq": 5, # Batas frekuensi minimum untuk lagu
}

In [18]:
# --- 1. Persiapan Device (CPU/GPU) ---
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU terdeteksi: {torch.cuda.get_device_name(0)}. Pelatihan akan menggunakan GPU.")
else:
    device = torch.device("cpu")
    print("GPU tidak ditemukan. Pelatihan akan menggunakan CPU.")

GPU terdeteksi: NVIDIA GeForce GTX 1650 Ti. Pelatihan akan menggunakan GPU.


In [19]:
# --- 2. Simulasi dan Pra-pemrosesan Data ---
def simulate_recsys_data():
    """Membuat DataFrame palsu untuk sistem rekomendasi."""
    print("\nMembuat simulasi dataset untuk Two-Tower Model...")
    num_samples = 20000
    num_unique_songs = 500
    
    songs = [f"Song Title {i+1}" for i in range(num_unique_songs)]
    song_ids = np.random.choice(songs, size=num_samples)
    
    messages = []
    for song in song_ids:
        song_num = int(song.split(" ")[-1])
        mood_type = (song_num % 4)
        if mood_type == 1:
            messages.append(f"aku cinta dia, seperti di lagu {song_num} ini")
        elif mood_type == 2:
            messages.append(f"hatiku hancur, rasanya sedih sekali, cocok dengan lagu {song_num}")
        elif mood_type == 3:
            messages.append(f"semangat pagi! hari ini cerah, ayo dengar lagu {song_num}")
        else:
            messages.append(f"aku marah dan kecewa, seperti lirik lagu {song_num}")
            
    data = {'message': messages, 'song_name': song_ids}
    return pd.DataFrame(data)

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def build_vocab(all_texts):
    word_counts = Counter(word for text in all_texts for word in text.split())
    vocab = {word: i+2 for i, (word, _) in enumerate(word_counts.most_common())}
    vocab['<PAD>'] = 0
    vocab['<UNK>'] = 1
    return vocab

In [20]:
# --- 3. Architecture Model ---

# Tower Pertama(message)
class MessageTower(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, config["text_embedding_dim"], padding_idx=0)
        self.gru = nn.GRU(
            config["text_embedding_dim"],
            config["hidden_dim"],
            num_layers=config["n_layers"],
            dropout=config["dropout"],
            batch_first=True
        )
        # Layer terakhir untuk memproyeksikan output GRU ke dimensi embedding akhir
        self.fc = nn.Linear(config["hidden_dim"], config["output_embedding_dim"])

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.gru(embedded)
        hidden = hidden[-1] # Ambil hidden state terakhir dari layer terakhir
        # Normalisasi L2 pada output embedding adalah praktik yang baik
        return F.normalize(self.fc(hidden), p=2, dim=1)

# Tower Kedua(song)
class SongTower(nn.Module):
    def __init__(self, num_songs, config):
        super().__init__()
        # Cukup sebuah layer embedding untuk merepresentasikan setiap lagu secara unik
        self.embedding = nn.Embedding(num_songs, config["output_embedding_dim"])

    def forward(self, x):
        # Normalisasi L2 pada output embedding
        return F.normalize(self.embedding(x), p=2, dim=1)

In [21]:
# --- 4. Dataset dan DataLoader ---
class RecommendationDataset(Dataset):
    def __init__(self, messages, song_data, vocab, song_map, max_len):
        self.messages = messages
        self.song_data = song_data # Berisi nama lagu (string)
        self.vocab = vocab
        self.song_map = song_map
        self.max_len = max_len

    def __len__(self):
        return len(self.messages)

    def __getitem__(self, idx):
        # Ambil pasangan (pesan, lagu) yang positif
        message = self.messages[idx]
        song_info = self.song_data[idx]  # FIX: gunakan string, jangan tuple
        
        # Vectorize pesan
        vectorized_msg = [self.vocab.get(word, self.vocab['<UNK>']) for word in message.split()]
        if len(vectorized_msg) < self.max_len:
            vectorized_msg += [self.vocab['<PAD>']] * (self.max_len - len(vectorized_msg))
        else:
            vectorized_msg = vectorized_msg[:self.max_len]
        
        # Dapatkan ID lagu dari map
        song_id = self.song_map[song_info]
        
        return torch.tensor(vectorized_msg, dtype=torch.long), torch.tensor(song_id, dtype=torch.long)

In [22]:
# --- 5. Fungsi Pelatihan ---
def train_recommendation_model(message_tower, song_tower, train_loader, config, device):
    message_tower.to(device)
    song_tower.to(device)
    
    # Gabungkan parameter dari kedua model untuk optimizer
    optimizer = optim.AdamW(
        list(message_tower.parameters()) + list(song_tower.parameters()),
        lr=config["learning_rate"]
    )
    # Loss function yang dirancang untuk tugas ini
    criterion = nn.CosineEmbeddingLoss(margin=config["margin"])

    for epoch in range(1, config["epochs"] + 1):
        start_time = time.time()
        message_tower.train()
        song_tower.train()
        total_loss = 0
        
        for messages, positive_songs in train_loader:
            messages, positive_songs = messages.to(device), positive_songs.to(device)
            
            optimizer.zero_grad()
            
            # Dapatkan embeddings dari kedua menara
            message_embeddings = message_tower(messages)
            song_embeddings = song_tower(positive_songs)
            
            # --- Strategi In-Batch Negative Sampling ---
            # Kita gunakan lagu dari item lain di batch yang sama sebagai sampel negatif.
            # Ini sangat efisien.
            
            # Loss untuk pasangan positif (target = 1)
            loss_pos = criterion(message_embeddings, song_embeddings, torch.ones(messages.size(0)).to(device))
            
            # Loss untuk pasangan negatif (target = -1)
            # Kita pasangkan message i dengan song j, di mana i != j
            # Cara termudah adalah dengan menggeser tensor lagu
            negative_song_embeddings = torch.roll(song_embeddings, shifts=1, dims=0)
            loss_neg = criterion(message_embeddings, negative_song_embeddings, -torch.ones(messages.size(0)).to(device))
            
            # Loss total adalah jumlah dari keduanya
            loss = loss_pos + loss_neg
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        end_time = time.time()
        print(f"Epoch: {epoch}/{config['epochs']} | "
              f"Waktu: {end_time - start_time:.2f}s | "
              f"Average Loss: {avg_loss:.4f}")

In [31]:
# --- 6. Fungsi untuk Rekomendasi (Inference) ---
def build_song_index(song_tower, song_map, device):
    """Proses semua lagu untuk membuat matriks embedding (indeks)."""
    print("\nMembangun indeks embedding untuk semua lagu...")
    song_tower.eval()
    with torch.no_grad():
        # Buat tensor berisi semua ID lagu
        all_song_ids = torch.arange(len(song_map)).to(device)
        # Dapatkan embedding untuk semua lagu sekaligus
        song_index_embeddings = song_tower(all_song_ids)
    print("Indeks berhasil dibangun.")
    return song_index_embeddings

def recommend_songs(message_text, message_tower, song_index, inv_song_map, vocab, config, device, top_k=10):
    """Memberikan top_k rekomendasi lagu untuk sebuah pesan."""
    message_tower.eval()
    
    # 1. Proses pesan input menjadi embedding
    cleaned_text = clean_text(message_text)
    vectorized = [vocab.get(word, vocab['<UNK>']) for word in cleaned_text.split()]
    if len(vectorized) < config["max_len"]:
        vectorized += [vocab['<PAD>']] * (config["max_len"] - len(vectorized))
    else:
        vectorized = vectorized[:config["max_len"]]
        
    query_tensor = torch.tensor(vectorized, dtype=torch.long).unsqueeze(0).to(device)
    
    with torch.no_grad():
        query_embedding = message_tower(query_tensor)
        
    # 2. Hitung kemiripan (cosine similarity) antara query dan semua lagu di indeks
    similarities = F.cosine_similarity(query_embedding, song_index)
    
    # 3. Dapatkan top_k lagu dengan similarity tertinggi
    top_results = torch.topk(similarities, k=top_k)
    
    # 4. Kembalikan nama lagu dan skornya
    recommendations = []
    for score, song_idx in zip(top_results.values, top_results.indices):
        song_name = inv_song_map[song_idx.item()]
        recommendations.append({
            "song": song_name,
            "similarity_score": score.item()
        })
    return recommendations

# MAIN EXECUTION


In [ ]:
# --- Pra-pemrosesan Data Asli ---
# Hapus baris dengan nilai kosong di kolom penting
df.dropna(subset=['message', 'song_name', 'song_artist'], inplace=True)
# Hapus pesan atau nama lagu yang hanya berisi spasi
df = df[df['message'].str.strip() != '']
df = df[df['song_name'].str.strip() != '']
df.reset_index(drop=True, inplace=True)

# Filter lagu yang jarang muncul untuk mengurangi noise
song_counts = df['song_name'].value_counts()
frequent_songs = song_counts[song_counts >= CONFIG["min_song_freq"]].index
df = df[df['song_name'].isin(frequent_songs)].reset_index(drop=True)

print(f"Dataset dimuat dan dibersihkan. Jumlah data setelah filtering: {len(df)}")

df['cleaned_message'] = df['message'].apply(clean_text)

Dataset dimuat dan dibersihkan. Jumlah data setelah filtering: 45041


In [25]:
# 2. Build Vocab dan Song Mapping
vocab = build_vocab(df['cleaned_message'])
all_songs = df['song_name'].unique().tolist()
song_map = {song: i for i, song in enumerate(all_songs)}
inv_song_map = {i: song for song, i in song_map.items()}

vocab_size = len(vocab)
num_songs = len(all_songs)
print(f"\nUkuran Vocabulary: {vocab_size}, Jumlah Lagu Unik: {num_songs}")


Ukuran Vocabulary: 83191, Jumlah Lagu Unik: 1334


In [26]:
# 3. Create Dataset & DataLoader
X = df['cleaned_message'].values
y = df['song_name'].values
# Tidak perlu split train/val untuk demo ini, tapi di dunia nyata WAJIB
train_dataset = RecommendationDataset(X, y, vocab, song_map, CONFIG["max_len"])
train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True)

In [27]:
# 4. Inisialisasi Model
message_tower = MessageTower(vocab_size, CONFIG)
song_tower = SongTower(num_songs, CONFIG)

In [28]:
# 5. Latih Model
train_recommendation_model(message_tower, song_tower, train_loader, CONFIG, device)

Epoch: 1/25 | Waktu: 18.54s | Average Loss: 0.8860
Epoch: 2/25 | Waktu: 17.71s | Average Loss: 0.8210
Epoch: 2/25 | Waktu: 17.71s | Average Loss: 0.8210
Epoch: 3/25 | Waktu: 17.75s | Average Loss: 0.7850
Epoch: 3/25 | Waktu: 17.75s | Average Loss: 0.7850
Epoch: 4/25 | Waktu: 17.88s | Average Loss: 0.7597
Epoch: 4/25 | Waktu: 17.88s | Average Loss: 0.7597
Epoch: 5/25 | Waktu: 17.60s | Average Loss: 0.7376
Epoch: 5/25 | Waktu: 17.60s | Average Loss: 0.7376
Epoch: 6/25 | Waktu: 17.64s | Average Loss: 0.7187
Epoch: 6/25 | Waktu: 17.64s | Average Loss: 0.7187
Epoch: 7/25 | Waktu: 17.61s | Average Loss: 0.6956
Epoch: 7/25 | Waktu: 17.61s | Average Loss: 0.6956
Epoch: 8/25 | Waktu: 17.60s | Average Loss: 0.6759
Epoch: 8/25 | Waktu: 17.60s | Average Loss: 0.6759
Epoch: 9/25 | Waktu: 17.61s | Average Loss: 0.6594
Epoch: 9/25 | Waktu: 17.61s | Average Loss: 0.6594
Epoch: 10/25 | Waktu: 17.60s | Average Loss: 0.6382
Epoch: 10/25 | Waktu: 17.60s | Average Loss: 0.6382
Epoch: 11/25 | Waktu: 17.93s 

In [29]:
# 6. Bangun Indeks untuk Inference
song_index = build_song_index(song_tower, song_map, device)


Membangun indeks embedding untuk semua lagu...
Indeks berhasil dibangun.


In [32]:
# 7. Tes Rekomendasi
print("\n--- Tes Rekomendasi ---")
test_message_1 = "kamu percaya ga sama aku? kamu tau ga sih bagi aku, kamu itu apa"
test_message_2 = "lu gila banget!"

recs_1 = recommend_songs(test_message_1, message_tower, song_index, inv_song_map, vocab, CONFIG, device)
print(f"\nRekomendasi untuk pesan: '{test_message_1}'")
for rec in recs_1:
   print(f"  - {rec['song']} (Score: {rec['similarity_score']:.4f})")
   
recs_2 = recommend_songs(test_message_2, message_tower, song_index, inv_song_map, vocab, CONFIG, device)
print(f"\nRekomendasi untuk pesan: '{test_message_2}'")
for rec in recs_2:
   print(f"  - {rec['song']} (Score: {rec['similarity_score']:.4f})")


--- Tes Rekomendasi ---

Rekomendasi untuk pesan: 'kamu percaya ga sama aku? kamu tau ga sih bagi aku, kamu itu apa'
  - Lucky (Score: 0.9945)
  - Goyang Dumang (Score: 0.9936)
  - So Long, London (Score: 0.9839)
  - pink skies (Score: 0.9769)
  - Fireflies (Score: 0.8008)
  - I JUST WANT TO KISS YOU NOW (Score: 0.7623)
  - Untungnya, Hidup Harus Tetap Berjalan (Score: 0.7499)
  - Birthday Sex (Score: 0.7465)
  - Sleep Well (Score: 0.7382)
  - Babaero (Score: 0.6761)

Rekomendasi untuk pesan: 'lu gila banget!'
  - Fuck You (Score: 0.9947)
  - stranger (Score: 0.9943)
  - Here, There And Everywhere - 2022 Mix (Score: 0.9440)
  - Sorry (Score: 0.8133)
  - HAHAHA (Score: 0.7873)
  - Anjing (Score: 0.7838)
  - My Favourite Clothes (Score: 0.7544)
  - Fool (Score: 0.7431)
  - Kepada Noor (Score: 0.7078)
  - Disini ngentot disana ngentod (Score: 0.6991)

Rekomendasi untuk pesan: 'kamu percaya ga sama aku? kamu tau ga sih bagi aku, kamu itu apa'
  - Lucky (Score: 0.9945)
  - Goyang Dumang (S

In [ ]:
# Ekspor model setelah training
# torch.save({
#     'message_tower_state_dict': message_tower.state_dict(),
#     'song_tower_state_dict': song_tower.state_dict(),
#     'vocab': vocab,
#     'song_map': song_map
# }, 'two_tower_songrec_model.pth')
# print("Model berhasil diekspor ke two_tower_songrec_model.pth")


Model berhasil diekspor ke two_tower_songrec_model.pth
